In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import geopandas as gpd
import sys
from sklearn.metrics import precision_recall_curve, auc
# ---------------------------------------------------------------------
# Add project root to Python path
# ---------------------------------------------------------------------


# project root = two levels above notebooks
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.data.data_loading import (
    load_full_dataset,
    load_full_training_pool,
    load_test_data,
    load_training_data,
    load_validation_data
)
import xgboost as xgb

In [3]:
from src.modeling.baseline import baseline_using_measurement_proportion, baseline_using_mean_radon_level

print(f"AUCPR for the baseline model that uses measurement proportion exceeding 200"
      f": {baseline_using_measurement_proportion().classification.mean()}")

print(f"AUCPR for the baseline model that uses mean radon level per FSA exceeding 200"
      f": {baseline_using_mean_radon_level().classification.mean()}")

AUCPR for the baseline model that uses measurement proportion exceeding 200: 0.023880597014925373
AUCPR for the baseline model that uses mean radon level per FSA exceeding 200: 0.06069651741293532


In [4]:
train_df = load_training_data(cv_fold=0)
val_df = load_validation_data(cv_fold=0)

In [5]:
print(f"Length of train: {len(train_df)}")
print(f"Length of train: {len(val_df)}")
print(f"Unique FSAs in train: {train_df['FSA'].nunique()}")
print(f"Unique FSAs in val: {val_df['FSA'].nunique()}")

Length of train: 8392
Length of train: 2069
Unique FSAs in train: 663
Unique FSAs in val: 191


In [6]:
removed_columns = ['spatial_cluster', 'is_test', 'cv_fold']
train_df = train_df.drop(columns= removed_columns)
val_df = val_df.drop(columns=removed_columns)

In [ ]:
CUTOFF_THRESHOLD = 0.5
def add_y_binary(df):
    if 'concentration' not in df.columns:
        raise ValueError("Dataframe must have the column `concentration`")
    df['y_binary'] = (df['concentration'] >200).astype(int)
    return df

def fsa_features_groupby(df):
    if "y_binary" not in df.columns:
        df = add_y_binary(df)
    fsa_features = df.drop(columns="y_binary").drop_duplicates("FSA")
    y_mean = df.groupby("FSA")["y_binary"].mean().reset_index(name="y_mean")
    return fsa_features.merge(y_mean, on="FSA")

def validation_X_y(df, approach = "Naive", threshold = CUTOFF_THRESHOLD):
    fsa_features = fsa_features_groupby(df)
    y_val = (fsa_features['y_mean'] > threshold).astype(int)
    if approach == "Naive":
        X_val = fsa_features.drop(
            columns=['concentration', 'provinceterritory', 'FSA', 'geometry', 'y_mean']
            )
        return (X_val,y_val)
    elif approach == "highly_correlated_only":
        columns_to_keep = ['hous_frac_type_single_detached', 'mean_uranium', 'geolprov_interior_platform',
                           'socioeco_frac_housing_burden', 'hous_median_value', 'hous_frac_type_other_attached']
        columns_to_drop = fsa_features.columns.difference(columns_to_keep)
        X_val = fsa_features.drop(columns = columns_to_drop)
        return (X_val, y_val)

def train_X_y(df, approach = "Naive"):
    if approach == "Naive":
        X = df.drop(columns=['concentration', 'y_binary', 'provinceterritory', 'FSA', 'geometry'])
        y = df['y_binary']
        return (X,y)
    ## Later when we try sophisticated approaches
    elif approach == "highly_correlated_only":
        columns_to_keep = ['hous_frac_type_single_detached', 'mean_uranium', 'geolprov_interior_platform',
                           'socioeco_frac_housing_burden', 'hous_median_value', 'hous_frac_type_other_attached']
        columns_to_drop = df.columns.difference(columns_to_keep)
        X = df.drop(columns= columns_to_drop)
        y = df['y_binary']
        return (X,y)

In [ ]:
pos_count = add_y_binary(train_df)['y_binary'].sum()
neg_count = len(train_df) - pos_count
print(f"Data positive to negative ratio: {pos_count/neg_count}")

max_depths = [1, 2, 3, 5, 10]
learning_rates = [0.05, 0.1, 0.5]

for max_depth in max_depths:
    for lr in learning_rates:
        clf = xgb.XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="aucpr",
            scale_pos_weight=neg_count / pos_count,
            max_depth= max_depth,
            n_estimators=300,
            learning_rate= lr ,
            random_state=42
        )
        clf.fit(
            train_X_y(train_df)[0], 
            train_X_y(train_df)[1],
            eval_set = [validation_X_y(val_df)],
            verbose = False
            )
        y_true = validation_X_y(val_df)[1]
        y_pred = clf.predict(validation_X_y(val_df)[0])

        precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
        auc_pr = auc(recall, precision)
        print(f"AUCPR for naive xgboost with max_depth {max_depth} and learning rate {lr}: {auc_pr:.2f}.")

Data positive to negative ratio: 0.1267454350161117
AUCPR for naive xgboost with max_depth 1 and learning rate 0.05: 0.34.
AUCPR for naive xgboost with max_depth 1 and learning rate 0.1: 0.34.
AUCPR for naive xgboost with max_depth 1 and learning rate 0.5: 0.34.
AUCPR for naive xgboost with max_depth 2 and learning rate 0.05: 0.34.
AUCPR for naive xgboost with max_depth 2 and learning rate 0.1: 0.24.
AUCPR for naive xgboost with max_depth 2 and learning rate 0.5: 0.25.
AUCPR for naive xgboost with max_depth 3 and learning rate 0.05: 0.24.
AUCPR for naive xgboost with max_depth 3 and learning rate 0.1: 0.13.
AUCPR for naive xgboost with max_depth 3 and learning rate 0.5: 0.01.
AUCPR for naive xgboost with max_depth 5 and learning rate 0.05: 0.25.
AUCPR for naive xgboost with max_depth 5 and learning rate 0.1: 0.15.
AUCPR for naive xgboost with max_depth 5 and learning rate 0.5: 0.01.
AUCPR for naive xgboost with max_depth 10 and learning rate 0.05: 0.01.
AUCPR for naive xgboost with max

In [9]:
# from sklearn.metrics import mean_absolute_percentage_error
# y_true = fsa_features_groupby(val_df).y_mean.tolist()
# y_pred = clf.predict_proba(validation_X_y(val_df)[0])[:,1]
# print(f"MAPE of predicted proportion above concentration >200: {mean_absolute_percentage_error(y_true,y_pred)}")

In [12]:
max_depths = [1, 3, 5, 10]
learning_rates = [0.05, 0.1, 0.5]
for max_depth in max_depths:
    for lr in learning_rates:
        clf = xgb.XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="aucpr",
            scale_pos_weight=neg_count / pos_count,
            max_depth= max_depth,
            n_estimators=300,
            learning_rate= lr ,
            random_state=42
        )
        clf.fit(
            train_X_y(train_df, approach= 'highly_correlated_only')[0], 
            train_X_y(train_df, approach= 'highly_correlated_only')[1],
            eval_set = [validation_X_y(val_df, approach= 'highly_correlated_only')],
            verbose = False
            )
        y_true = validation_X_y(val_df, approach= 'highly_correlated_only')[1]
        y_pred = clf.predict(validation_X_y(val_df, approach= 'highly_correlated_only')[0])

        precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
        auc_pr = auc(recall, precision)
        print(f"AUCPR for xgbooast with highly correlated features only with max_depth {max_depth} and learning rate {lr}: {auc_pr:.2f}.")

AUCPR for xgbooast with highly correlated features only with max_depth 1 and learning rate 0.05: 0.23.
AUCPR for xgbooast with highly correlated features only with max_depth 1 and learning rate 0.1: 0.24.
AUCPR for xgbooast with highly correlated features only with max_depth 1 and learning rate 0.5: 0.12.
AUCPR for xgbooast with highly correlated features only with max_depth 3 and learning rate 0.05: 0.24.
AUCPR for xgbooast with highly correlated features only with max_depth 3 and learning rate 0.1: 0.24.
AUCPR for xgbooast with highly correlated features only with max_depth 3 and learning rate 0.5: 0.13.
AUCPR for xgbooast with highly correlated features only with max_depth 5 and learning rate 0.05: 0.01.
AUCPR for xgbooast with highly correlated features only with max_depth 5 and learning rate 0.1: 0.01.
AUCPR for xgbooast with highly correlated features only with max_depth 5 and learning rate 0.5: 0.01.
AUCPR for xgbooast with highly correlated features only with max_depth 10 and l